In [11]:
print(123)

123


In [12]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")


In [13]:
df_ground_truth.head()

,question,document
0,Can I enroll in this course if I'm just findin...,74eb249bbf
1,What's the deadline for submitting my project ...,74eb249bbf
2,Do I need to start the course at a specific ti...,74eb249bbf
3,"If I join late, am I still able to complete ev...",74eb249bbf
4,Is there a cutoff date for when I have to turn...,74eb249bbf


In [14]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [15]:
ground_truth[10]

{'question': 'Where can I find the link to watch the office hours live?',
 'document': '489dd1c9d9'}

In [16]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [17]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [18]:
text_search("What is the difference between LLM and traditional machine learning?")

[{'id': '57c8d3a1e4',
  'course': 'llm-zoomcamp',
  'section': 'Module 4: Evaluation',
  'question': 'Why is there a gap between lessons 6 and 11 in Module 4?',
  'answer': 'No lessons are missing. Module 4 is split into two subsections, and the filenames reflect that structure:\n\n- `01`–`06`: RAG evaluation\n- `11`–`15`: agent evaluation\n\nYou can read the numbering as lessons 0.1–0.6 for the first subsection and 1.1–1.5 for the second. That is why the filenames jump from `06-search-tuning.md` to `11-evaluation-intro.md`.'},
 {'id': 'dbf5369006',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Can I use Bluesky for learning in public credits?',
  'answer': 'Yes. Bluesky posts can be used for learning in public credits.'},
 {'id': 'db78580409',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'What is the cosine similarity?',
  'answer': 'Cosine similarity is a measure used to calculate the similarity betwee

In [19]:
q = ground_truth[10]
q

{'question': 'Where can I find the link to watch the office hours live?',
 'document': '489dd1c9d9'}

In [22]:
doc_id = q['document']
doc_id

'489dd1c9d9'

In [23]:
results = text_search(q['question'])
results

[{'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'},
 {'id': 'd65e05bd7a',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Are there live sessions or office hours for each module?',
  'answer': 'There are no separate live sessions for every module by default. Module materials are pre-recorded and

In [24]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

489dd1c9d9 == 489dd1c9d9: True
d65e05bd7a == 489dd1c9d9: False
53f15299b6 == 489dd1c9d9: False
930286278d == 489dd1c9d9: False
5cc511f85b == 489dd1c9d9: False


In [25]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [26]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [27]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [28]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [31]:
relevance_total_text[:15]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [37]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance 

In [38]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [39]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [40]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/765 [00:00<?, ?it/s]

In [42]:
relevance_total[15:50]

[[0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]